# Example 12.6.6 / 2: Trade mean loss against CVaR

Companion notebook for **12.6.6 Risk Measures and CVaR** in the AMPL lab Documentation, section "Trade mean loss against CVaR". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Trade mean loss against CVaR

Plan A has the losses above (mean 40, CVaR at 0.8 equal to 100). Plan B has losses
`35, 35, 40, 45, 60` (mean 43, CVaR 60). `Mix` is the share of plan A. The run solves the
`MeanRisk` objective for three weights, then sets `risk_weight` to 0 and imposes
`CVaR <= risk_limit` with a limit of 80.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SCENARIOS;                          # loss scenarios
param probability {SCENARIOS} >= 0;     # scenario weight
param loss_a {SCENARIOS};               # plan A loss, thousand dollars
param loss_b {SCENARIOS};               # plan B loss, thousand dollars
param alpha > 0 < 1;                    # confidence level of the tail
param risk_weight >= 0;                 # price of tail risk
param risk_limit default Infinity;      # CVaR cap, thousand dollars

var Mix >= 0 <= 1;                      # share of plan A
var eta;                                # VaR candidate
var excess {SCENARIOS} >= 0;            # loss above eta

# loss of the mix, thousand dollars
var Loss {s in SCENARIOS} = Mix * loss_a[s] + (1 - Mix) * loss_b[s];
var ExpectedLoss = sum {s in SCENARIOS} probability[s] * Loss[s];  # mean loss, thousand dollars
var CVaR = eta                          # CVaR at level alpha, thousand dollars
    + sum {s in SCENARIOS} probability[s] * excess[s] / (1 - alpha);

subject to TailLoss {s in SCENARIOS}:   # excess is at least the loss above eta
    excess[s] >= Loss[s] - eta;

subject to RiskLimit: CVaR <= risk_limit;  # inactive while risk_limit is Infinity

minimize MeanRisk:                      # expected loss plus weighted CVaR
    ExpectedLoss + risk_weight * CVaR;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
scenarios = ["s1", "s2", "s3", "s4", "s5"]
ampl.set["SCENARIOS"] = scenarios
ampl.param["probability"] = {s: 0.2 for s in scenarios}
ampl.param["loss_a"] = dict(zip(scenarios, [10, 20, 30, 40, 100]))
ampl.param["loss_b"] = dict(zip(scenarios, [35, 35, 40, 45, 60]))
ampl.param["alpha"] = 0.8

In [5]:
# run.py
def report(label):
    ampl.solve(solver="highs", verbose=False)
    print(label, ampl.solve_result, round(ampl.var["Mix"].value(), 6),
          round(ampl.var["ExpectedLoss"].value(), 6), round(ampl.var["CVaR"].value(), 6))

for weight in [0, 0.05, 0.1]:
    ampl.param["risk_weight"] = weight
    report(f"weight {weight}:")

ampl.param["risk_weight"] = 0
ampl.param["risk_limit"] = 80
report("limit 80:")

weight 0: solved 1.0 40.0 100.0
weight 0.05: solved 1.0 40.0 100.0
weight 0.1: solved 0.0 43.0 60.0
limit 80: solved 0.5 41.5 80.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set SCENARIOS := s1 s2 s3 s4 s5;
param probability default 0.2;
param:  loss_a  loss_b :=
  s1        10      35
  s2        20      35
  s3        30      40
  s4        40      45
  s5       100      60;
param alpha := 0.8;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
for {weight in {0, 0.05, 0.1}} {
    let risk_weight := weight;
    solve;
    display risk_weight, Mix, ExpectedLoss, CVaR;
}
let risk_weight := 0;
let risk_limit := 80;
solve;
display risk_limit, Mix, ExpectedLoss, CVaR;

risk_weight = 0
Mix = 1
ExpectedLoss = 40
CVaR = 100

risk_weight = 0.05
Mix = 1
ExpectedLoss = 40
CVaR = 100

risk_weight = 0.1
Mix = 0
ExpectedLoss = 43
CVaR = 60

risk_limit = 80
Mix = 0.5
ExpectedLoss = 41.5
CVaR = 80

